# Investment Analysis (using the `investment_analysis` package)

This notebook is now just a thin wrapper: all the real logic lives in the `investment_analysis/` package so it can be reused, tested, and imported elsewhere.

In [ ]:
import sys
sys.path.append('..')  # so we can import the package and config from the project root

import numpy as np
from config import AnalysisConfig
from investment_analysis import data, returns, risk_metrics, optimization, visualization

cfg = AnalysisConfig()
weights = np.array(cfg.weights)

In [ ]:
prices = data.download_prices(cfg.tickers, cfg.start_date, cfg.end_date)
benchmark_prices = data.download_benchmark(cfg.benchmark_ticker, cfg.start_date, cfg.end_date)
risk_free_rate = data.download_risk_free_rate(cfg.risk_free_ticker)
prices

In [ ]:
visualization.plot_prices(prices)

In [ ]:
daily_returns = returns.simple_returns(prices)
log_return_df = returns.log_returns(prices)
benchmark_daily_returns = benchmark_prices.pct_change().dropna()

portfolio_daily_returns = returns.portfolio_returns(daily_returns, weights)
annual_return = returns.annualize_simple_return(portfolio_daily_returns.mean())
annual_volatility = returns.annualize_volatility(np.std(portfolio_daily_returns))
annual_return, annual_volatility

In [ ]:
port_arr = portfolio_daily_returns.to_numpy().flatten()
bench_arr = benchmark_daily_returns.to_numpy().flatten()

port_beta = risk_metrics.beta(port_arr, bench_arr)
port_alpha = risk_metrics.alpha(portfolio_daily_returns, benchmark_daily_returns, port_beta, risk_free_rate)
sharpe = risk_metrics.sharpe_ratio(annual_return, risk_free_rate, annual_volatility)

port_beta, port_alpha, sharpe

In [ ]:
expected_returns, cov_matrix_annual = optimization.annualized_inputs(log_return_df)
pm = optimization.PortfolioMath(expected_returns, cov_matrix_annual, risk_free_rate)

sim_df = optimization.monte_carlo_portfolios(pm, cfg.n_monte_carlo_portfolios, seed=cfg.random_seed)
min_vol = optimization.minimum_volatility_portfolio(pm, cfg.tickers)
max_sharpe = optimization.maximum_sharpe_portfolio(pm, cfg.tickers)

target_returns = np.linspace(sim_df['Return'].min(), sim_df['Return'].max(), cfg.n_frontier_points)
frontier_df = optimization.efficient_frontier(pm, target_returns)

In [ ]:
visualization.plot_efficient_frontier(
    sim_df, frontier_df,
    max_sharpe_point=(max_sharpe.volatility, max_sharpe.expected_return),
    min_vol_point=(min_vol.volatility, min_vol.expected_return),
)